In [1]:
import os
from pathlib import Path

import polars as pl
import pandas as pd
import numpy as np

import catboost as cb
import lightgbm as lgb
import xgboost as xgb

import matplotlib.pyplot as plt
import seaborn as sns

from mllabs import Experimenter, Connector, ColSelector, ProgressSessionLogger, TqdmProgressSession
from mllabs.adapter import XGBoostAdapter, LightGBMAdapter, CatBoostAdapter
from mllabs.nn import NNClassifier
from mllabs.col import ohe_drop_first
from mllabs.collector import SHAPCollector
from mllabs.filter import RandomFilter
from mllabs.processor import PolarsLoader, PandasConverter, ExprProcessor
from mllabs.collector import MetricCollector, ModelAttrCollector

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.metrics import balanced_accuracy_score

2026-05-12 00:17:03.846392: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-12 00:17:04.147538: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-12 00:17:16.105469: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
/home/sun9sun9/python312/lib/python3.12/site-packages/keras/src/export/tf2onnx_lib.py:8: Fu

In [2]:
data_path = Path('data')

dict_expr = {
    'Mulching_Used_n': (pl.col('Mulching_Used') == 'Yes').cast(pl.Int8),
    'Soil_25': (pl.col('Soil_Moisture') < 25).cast(pl.Int8),
    'Temp_30': (pl.col('Temperature_C') > 30).cast(pl.Int8),
    'Rain_300': (pl.col('Rainfall_mm') < 300).cast(pl.Int8),
    'Wind_10': (pl.col('Wind_Speed_kmh') > 10).cast(pl.Int8),
}

loader = make_pipeline(
    PolarsLoader(predefined_types={'id': pl.Int64}),
    ExprProcessor(dict_expr=dict_expr),
)

df_train = loader.fit_transform([data_path / 'train.csv'])
df_test = loader.transform([data_path / 'test.csv'])

y = 'Irrigation_Need'
X_cat = ['Crop_Growth_Stage', 'Crop_Type', 'Irrigation_Type', 'Region', 'Season', 'Soil_Type', 'Water_Source']
X_num = ['Electrical_Conductivity', 'Field_Area_hectare', 'Humidity', 'Organic_Carbon', 'Previous_Irrigation_mm', 'Rainfall_mm', 
         'Soil_Moisture', 'Soil_pH', 'Sunlight_Hours', 'Temperature_C','Wind_Speed_kmh']
X_bin = ['Mulching_Used_n']
X_num_bin = ['Soil_25', 'Temp_30', 'Rain_300', 'Wind_10']
X_all = X_num + X_cat + X_bin + X_num_bin

y_repl = {'Low': 0, 'Medium': 1, 'High': 2}
y_weight = {'Low': 1.000000, 'Medium': 1.547291, 'High': 17.607549}
df_train = df_train.with_columns(
    **{
        y: pl.col(y).replace(y_repl).cast(pl.Int8),
        'sample_weight': pl.col(y).cast(pl.String).replace(y_weight).cast(pl.Float32)
    }
)

In [3]:
# rm -rf exp/fe

In [3]:
if not os.path.exists('exp/fe'):
    sss1 = StratifiedShuffleSplit(n_splits = 1, random_state = 123, train_size = 0.8)
    sss2 = StratifiedShuffleSplit(n_splits = 1, random_state = 123, train_size = 0.9)
    e_fe = Experimenter.create(
        df_train, 'exp/fe', sp = sss1, sp_v = sss2, splitter_params={'y': y}, 
        logger=ProgressSessionLogger(level=['info', 'progress'], session_cls=TqdmProgressSession))
    e_fe.set_grp('clf', method = 'predict', role = 'head', edges = {'y': [(None, y)], 'sample_weight': [(None, 'sample_weight')]})
    e_fe.set_grp(
        'xgb', parent = 'clf', processor=xgb.XGBClassifier, adapter=XGBoostAdapter(eval_mode='both'), 
        params={'random_state': 123, 'n_estimators': 10000, 'learning_rate': 0.05, 'enable_categorical': True, 'early_stopping_rounds': 50, 'eval_metric': 'mlogloss'})
    e_fe.add_collector(ModelAttrCollector('xgb_evals_results', Connector(processor=xgb.XGBClassifier), 'evals_result'))
    e_fe.set_grp(
        'lgb', parent = 'clf', processor=lgb.LGBMClassifier, adapter=LightGBMAdapter(eval_mode='both'), 
        params={'random_state': 123, 'n_estimators': 10000, 'learning_rate': 0.05, 'verbose': -1, 'early_stopping': {'stopping_rounds': 50, 'first_metric_only': True},'eval_metric': 'multi_logloss'})
    e_fe.add_collector(
        ModelAttrCollector('lgb_evals_results', Connector(processor=lgb.LGBMClassifier), 'evals_result'))
    e_fe.set_grp(
        'cb', parent = 'clf', processor=cb.CatBoostClassifier, adapter=CatBoostAdapter(eval_mode='valid'),
        params={'early_stopping_rounds': 50, 'verbose': 0, 'random_state': 123, 'cat_features': ColSelector(col_type='category')})
    e_fe.add_collector(
        ModelAttrCollector('cb_evals_results', Connector('_base$', processor=cb.CatBoostClassifier), 'evals_result'))
    e_fe.set_grp('nn', parent = 'clf', processor = NNClassifier, params = {'metrics': ['sparse_categorical_crossentropy'], 'early_stopping': 10})
    e_fe.add_collector(
        ModelAttrCollector('nn_evals', Connector(processor=NNClassifier), result_key='evals_result'))
    e_fe.set_grp('lr', parent='clf', processor=LogisticRegression)

    e_fe.set_grp('pre', role = 'stage', method='transform')
    e_fe.set_grp('pre_ft', role = 'stage', method='fit_transform', edges = {'y': [(None, y)]})
    e_fe.set_node('std', grp='pre', processor=StandardScaler, edges={'X': [(None, X_num)]})
    e_fe.set_node('ohe', grp='pre', processor=OneHotEncoder, edges={'X': [(None, X_cat)]}, params={'sparse_output': False})
    e_fe.build()
    
    y_edges = {'y': [(None, y)]}
    e_fe.add_collector(
        MetricCollector('bAcc', Connector(edges = y_edges, role = 'head'), slice(-1, None), balanced_accuracy_score, include_train = True))
    e_fe.add_collector(
        ModelAttrCollector('lgb_feature_importance', Connector(processor=lgb.LGBMClassifier, edges=y_edges), 'feature_importances'))
    e_fe.add_collector(
        ModelAttrCollector(
            'xgb_feature_importance_gain', Connector(processor=xgb.XGBClassifier, edges=y_edges), 'feature_importances', params = {'importance_type': 'gain'}))
    e_fe.add_collector(
        ModelAttrCollector(
            'xgb_feature_importance_cover', Connector(processor=xgb.XGBClassifier, edges=y_edges), 'feature_importances', params = {'importance_type': 'cover'}))
    e_fe.add_collector(
        ModelAttrCollector('cb_feature_importance', Connector(processor=cb.CatBoostClassifier, edges=y_edges), 'feature_importances_pvc'))
    e_fe.add_collector(
        ModelAttrCollector('cb_interaction', Connector(processor=cb.CatBoostClassifier, edges=y_edges), 'feature_importances_interaction'))
    e_fe.add_collector(
        ModelAttrCollector('lr_coef', Connector(processor=LogisticRegression, edges=y_edges), 'coef'))
else:
    e_fe = Experimenter.load('exp/fe', df_train, logger=ProgressSessionLogger(level=['info', 'progress'], session_cls=TqdmProgressSession))

Loaded: 22 node(s), 8 group(s), 1 fold(s)


In [4]:
e_fe.set_node('xgb_base', grp='xgb', edges={'X': [(None, X_num + X_cat + X_bin)]}, params={'n_estimators': 10000, 'gpu': 'yes'})
e_fe.set_node('lgb_base', grp='lgb', edges={'X': [(None, X_num + X_cat + X_bin)]}, params={'n_estimators': 10000, 'gpu': None})
e_fe.set_node('cb_base', grp='cb', edges={'X': [(None, X_num + X_cat + ['Mulching_Used'])]}, params={'n_estimators': 10000, 'gpu': 'yes'})
e_fe.set_node('nn_base', grp='nn', edges={'X': [('std', None), ('ohe', ohe_drop_first), (None, X_bin)]}, params={'epochs': 200, 'gpu': 'yes'})
e_fe.set_node('lr_base', grp='lr', edges={'X': [('std', None), ('ohe', ohe_drop_first), (None, X_bin)]})
e_fe.exp(finalize=True, n_jobs=2, gpu_id_list=[0])

No head nodes to experiment


In [5]:
e_fe.get_collector('bAcc').get_metrics_agg('.*_base$')[0]

,test,train,valid
cb_base,0.965599,0.990882,0.964416
lgb_base,0.967129,0.994087,0.966176
lr_base,0.850281,0.849520,0.850366
nn_base,0.962443,0.974719,0.960274
xgb_base,0.967986,0.993891,0.966524


## X_num_bin 속성 활용

In [6]:
from mllabs.processor import TypeConverter, CatConverter
e_fe.set_node('X_num_bin2str', grp='pre', edges={'X': [(None, X_num_bin)]}, processor=TypeConverter, params={'to':'str'})
e_fe.set_node('X_num_binstr2cat', grp='pre', edges={'X': [('X_num_bin2str', None)]}, processor=CatConverter)
e_fe.build()

No stage nodes to build


In [7]:
e_fe.set_node('xgb_X_num_bin', grp='xgb', edges={'X': [(None, X_num + X_cat + X_bin + X_num_bin)]}, params={'n_estimators': 10000, 'gpu': 'yes'})
e_fe.set_node('lgb_X_num_bin', grp='lgb', edges={'X': [(None, X_num + X_cat + X_bin + X_num_bin)]}, params={'n_estimators': 10000, 'gpu': None})
e_fe.set_node('cb_X_num_bin', grp='cb', edges={'X': [(None, X_num + X_num_bin + X_cat + ['Mulching_Used'])]}, params={'n_estimators': 10000, 'gpu': 'yes'})
e_fe.set_node('cb_X_num_bin_2', grp='cb', edges={'X': [(None, X_num + X_cat + ['Mulching_Used']), ('X_num_binstr2cat', None)]}, params={'n_estimators': 10000, 'gpu': 'yes'})
e_fe.set_node('nn_X_num_bin', grp='nn', edges={'X': [('std', None), ('ohe', ohe_drop_first), (None, X_bin), (None, X_num_bin)]}, params={'epochs': 200, 'gpu': 'yes'})
e_fe.set_node('lr_X_num_bin', grp='lr', edges={'X': [('std', None), ('ohe', ohe_drop_first), (None, X_bin), (None, X_num_bin)]})
e_fe.set_node('lr_X_num_bin2', grp='lr', edges={'X': [('ohe', ohe_drop_first), (None, X_bin), (None, X_num_bin)]})
e_fe.set_node('lr_X_num_bin3', grp='lr', edges={'X': [('ohe', ('cap', ohe_drop_first, ['ohe__Crop_Growth_Stage.*'])), (None, X_bin), (None, X_num_bin)]})
e_fe.exp(finalize=True, n_jobs=2, gpu_id_list=[0])

No head nodes to experiment


In [8]:
e_fe.get_collector('bAcc').get_metrics_agg('.*X_num_bin.*')[0]

,test,train,valid
cb_X_num_bin,0.967261,0.991609,0.965281
cb_X_num_bin_2,0.967175,0.991748,0.965477
lgb_X_num_bin,0.967598,0.994852,0.965646
lr_X_num_bin,0.962480,0.962093,0.960545
lr_X_num_bin2,0.962407,0.962227,0.960209
lr_X_num_bin3,0.962681,0.962494,0.959773
nn_X_num_bin,0.964052,0.965845,0.961147
xgb_X_num_bin,0.968056,0.993922,0.966496


In [9]:
e_fe.get_collector('lr_coef').get_attr('lr_X_num_bin3')[0][0].unstack(level = 0)

,0,1,2
ohe__Crop_Growth_Stage_Harvest,11.177913,-2.177492,-9.000421
ohe__Crop_Growth_Stage_Sowing,11.342007,-1.961637,-9.380369
ohe__Crop_Growth_Stage_Vegetative,-0.000317,0.002413,-0.002095
Mulching_Used_n,6.130100,-0.777283,-5.352818
Soil_25,-11.717744,2.058867,9.658878
Temp_30,-6.001985,0.604587,5.397398
Rain_300,-10.676323,1.239224,9.437099
Wind_10,-5.766726,0.771829,4.994897
intercept,5.736239,3.482962,-9.219200


## Cross fit LogisticRegression

In [10]:
from mllabs.processor import CrossFitTransformer

In [11]:
e_fe.set_node('X_lr_cf', grp='pre_ft', edges={'X': [('ohe', ('cap', ohe_drop_first, ['ohe__Crop_Growth_Stage.*'])), (None, X_bin), (None, X_num_bin)]}, 
              processor=CrossFitTransformer, params={'estimator':LogisticRegression()})

{'result': 'skip',
 'affected_nodes': [],
 'old_obj': <mllabs._pipeline.PipelineNode at 0x7dec91112e10>,
 'obj': <mllabs._pipeline.PipelineNode at 0x7dec91112e10>}

In [12]:
e_fe.build()

No stage nodes to build


In [13]:
e_fe.set_node('xgb_X_lr_cf', grp='xgb', edges={'X': [(None, X_num + X_cat + X_bin + X_num_bin), ('X_lr_cf', None)]}, params={'n_estimators': 10000, 'gpu': 'yes'})
e_fe.set_node('lgb_X_lr_cf', grp='lgb', edges={'X': [(None, X_num + X_cat + X_bin + X_num_bin), ('X_lr_cf', None)]}, params={'n_estimators': 10000, 'gpu': None})
e_fe.set_node('cb_X_lr_cf', grp='cb', edges={'X': [(None, X_num + X_num_bin + X_cat + ['Mulching_Used']), ('X_lr_cf', None)]}, params={'n_estimators': 10000, 'gpu': 'yes'})
e_fe.set_node('nn_lr_cf', grp='nn', edges={'X': [('std', None), ('ohe', ohe_drop_first), (None, X_bin), (None, X_num_bin), ('X_lr_cf', slice(0, -1))]}, params={'epochs': 200, 'gpu': 'yes'})
e_fe.exp(finalize=True, n_jobs=2, gpu_id_list=[0])

No head nodes to experiment


In [14]:
e_fe.get_collector('bAcc').get_metrics_agg('.*_lr_cf.*')[0]

,test,train,valid
cb_X_lr_cf,0.967457,0.990869,0.965327
lgb_X_lr_cf,0.966675,0.993380,0.965073
nn_lr_cf,0.963505,0.965255,0.960364
xgb_X_lr_cf,0.968551,0.993142,0.965357


In [15]:
d = e_fe.get_collector('xgb_evals_results').get_attrs()

In [16]:
pd.concat([
    e_fe.collectors['xgb_feature_importance_gain'].get_attrs_agg('xgb_X_lr_cf').pipe(
        lambda x: x / x.sum()
    ).rename('xgb_gain'),
    e_fe.collectors['xgb_feature_importance_cover'].get_attrs_agg('xgb_X_lr_cf').pipe(
        lambda x: x / x.sum()
    ).rename('xgb_cover')
], axis=1).sort_values('xgb_gain', ascending = False).iloc[:10]

,xgb_gain,xgb_cover
X_lr_cf__logisticregression_0,0.441166,0.121246
X_lr_cf__logisticregression_1,0.378558,0.102865
X_lr_cf__logisticregression_2,0.096920,0.067911
Rainfall_mm,0.008530,0.070914
Rain_300,0.007242,0.128499
Temp_30,0.005768,0.049427
Soil_25,0.004389,0.039719
Temperature_C,0.004090,0.040896
Previous_Irrigation_mm,0.003891,0.040289
Soil_Moisture,0.003684,0.042937


In [17]:
lr_cf = e_fe.get_train_data({'X': [('X_lr_cf', None)], 'y': [(None, y)]})

In [18]:
from scipy.optimize import differential_evolution
def neg_balanced_accuracy(thresholds):
    adjusted = lr_cf['X'].data.to_numpy() * thresholds
    return -balanced_accuracy_score(lr_cf['y'].data, adjusted.argmax(axis=1))
result = differential_evolution(
    neg_balanced_accuracy, bounds=[(0, 3.0), (0, 3.0), (0, 3.0)],
    seed = 42, maxiter = 1000, tol = 1e-8
)

In [19]:
best_weights = result.x
optimized_cv = -result.fun
best_weights, optimized_cv

(array([0.57029392, 0.2418915 , 2.93817578]), np.float64(0.9619603070788134))